<a href="https://colab.research.google.com/github/MartyYano/patent-image-ocr-pipeline/blob/main/patent_ocr_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Linux側のPDF処理ツールを先に入れる ---
!apt-get update -qq
!apt-get install -y -qq poppler-utils

# --- Pythonライブラリをまとめて入れる ---
!pip install -q \
    pdf2image \
    pandas \
    chromadb \
    tiktoken \
    langchain-openai==0.3.33 \
    openai==1.106.1

# --- poppler / pdfinfo が使えるか確認 ---
!which pdfinfo
!pdfinfo -v

## PDFをPNG化 & 英文抽出

In [ ]:
# ============================================================
# Patent PDF High-Precision OCR & Translation Pipeline
# (英文スキャン特許PDF向け 高精度・高速・低コスト AI-OCRパイプライン)
#
# 開発者  : So Yano
# 最終更新: 2026-07-31 (v2.0)
#
# 【設計思想 & コアな強み】
# 推論型モデル（Reasoning Model）に頼る画像解析は、思考トークン消費による
# 「極端な処理遅延」と「高額なAPIコスト」が大きな課題となる。
# 本コードは、OpenCVによる適切な画像前処理と軽量・標準な「非推論型VLM (gpt-4.1)」を
# マルチスレッド並列パイプライン化することで、推論型モデルを使わずに
# 【圧倒的な高速化・低コスト・文字抽出精度ほぼ100%】を同時に達成。
#
# 【パイプライン構成】
#   1. pdf_to_image           : 400DPI変換, 左右2アップ分割, 余白トリミング, カラーPNG生成
#   2. Image_to_English_Text  : 非推論型VLM × 並列処理(ThreadPoolExecutor+Semaphore)による高速OCR
#   3. Claim_Extraction       : 特許請求項（Claims）の原文Markdown抽出
#   4. Claim_Translation      : 特許実務（US/JP）の法的ニュアンスを保持した日本語翻訳
#
# 【最適化・リファクタリング実績】
#   ・[FIX1] 不要ライブラリ（tiktoken）削除による環境軽量化
#   ・[FIX2] 関数設計の見直し（self非依存の独立モジュール化）
#   ・[FIX3] カラー情報を維持した前処理（VLMの文字認識精度向上）
#   ・[FIX4] CLAHE（適応的ヒストグラム均等化）のパラメータ最適化
#   ・[FIX5] 全工程のモデルを非推論型 gpt-4.1 へ統一（速度・コスト最適化）
#   ・[FIX6] APIパラメータ最適化（top_p削除による推論挙動の安定化）
#   ・[FIX7] APIリトライ構造（最大5回）実装による通信の堅牢化
#   ・[FIX8] トークン消費コスト最適化（無駄な補間拡大処理の廃止）
# ============================================================


# ============================================================
# Cell 1: 環境構築
# ============================================================

# !apt-get update -qq
# !apt-get install -y -qq poppler-utils
# !pip install -q \
#     pdf2image \
#     pandas \
#     chromadb \
#     langchain-openai==0.3.33 \
#     openai==1.106.1
# !which pdfinfo
# !pdfinfo -v


# ============================================================
# Cell 2: PDF → 画像変換 & OCR → テキスト保存
# ============================================================

from pdf2image import convert_from_path
from PIL import Image

import os
import time
import base64
import io

import numpy as np
import cv2

# [FIX1] tiktoken 削除（encoder は未使用のため不要）

from openai import (
    OpenAI,
    AzureOpenAI
)

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

from threading import Semaphore

from google.colab import drive
from google.colab import userdata

# Google Drive のマウント
drive.mount('/content/drive')

select = 1

pdf_files = {
    1: "./data/sample_patent_claim.pdf",
    2: "./data/sample_patent_01.pdf",
    3: "./data/sample_patent_02.pdf",
}

_fpath = pdf_files.get(select)
if not _fpath or not os.path.exists(_fpath):
    raise FileNotFoundError(f"選択したPDFファイルが見つかりません: {_fpath}")

# 保存先フォルダ
_save_folder = "./output"

t0 = time.perf_counter()


def pdf_to_image(_fpath):
    """
    PDFを画像に変換し、必要に応じて画質前処理を行い、
    カラーPNGのBase64文字列リストを返す。
    """

    # =========================================================
    # 設定
    # =========================================================
    _dpi = 400

    # 見開き2ページを左右に分割する場合はTrue
    _split_two_up = True

    # 上下のトリミング率
    _trim_top_pct = 0.02
    _trim_bottom_pct = 0.02

    # 以下の3設定は変更しない
    _do_deskew = False
    _do_clahe = False
    _do_unsharp = False

    # 傾き補正設定
    _deskew_max_angle = 30.0

    # CLAHE設定
    _tileGridSize = (6, 6)

    # アンシャープ設定
    _amount = 1.7
    _sigma = 0.9
    _threshold = 3

    # =========================================================
    # PIL画像からOpenCV画像へ変換
    # RGB → BGR
    # =========================================================
    def _pil_to_cv(img_pil):
        return cv2.cvtColor(
            np.array(img_pil),
            cv2.COLOR_RGB2BGR
        )

    # =========================================================
    # 上下左右の余白をトリミング
    # =========================================================
    def _trim_margins(
        img_pil,
        top_pct,
        bottom_pct,
        left_pct=0.0,
        right_pct=0.0
    ):
        w, h = img_pil.size

        top = int(h * top_pct)
        bottom = h - int(h * bottom_pct)
        left = int(w * left_pct)
        right = w - int(w * right_pct)

        return img_pil.crop(
            (left, top, right, bottom)
        )

    # =========================================================
    # 傾き補正
    # =========================================================
    def _deskew(
        img_cv,
        max_angle=_deskew_max_angle
    ):
        gray = cv2.cvtColor(
            img_cv,
            cv2.COLOR_BGR2GRAY
        )

        gray_inv = cv2.bitwise_not(gray)

        thresh = cv2.threshold(
            gray_inv,
            0,
            255,
            cv2.THRESH_BINARY | cv2.THRESH_OTSU
        )[1]

        coords = np.column_stack(
            np.where(thresh > 0)
        )

        if coords.size == 0:
            return img_cv

        angle = cv2.minAreaRect(coords)[-1]

        if angle < -45:
            angle = 90 + angle

        # 大きすぎる傾きは誤検出とみなす
        if abs(angle) > max_angle:
            return img_cv

        h, w = img_cv.shape[:2]

        rotation_matrix = cv2.getRotationMatrix2D(
            (w // 2, h // 2),
            -angle,
            1.0
        )

        return cv2.warpAffine(
            img_cv,
            rotation_matrix,
            (w, h),
            flags=cv2.INTER_CUBIC,
            borderMode=cv2.BORDER_REPLICATE
        )

    # =========================================================
    # CLAHE
    # L成分のみ補正し、カラー情報を維持
    # =========================================================
    def _clahe(img_cv):
        lab = cv2.cvtColor(
            img_cv,
            cv2.COLOR_BGR2LAB
        )

        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=_tileGridSize
        )

        l2 = clahe.apply(l)

        lab2 = cv2.merge(
            [l2, a, b]
        )

        return cv2.cvtColor(
            lab2,
            cv2.COLOR_LAB2BGR
        )

    # =========================================================
    # アンシャープ処理
    # 輝度成分のみシャープ化
    # =========================================================
    def _unsharp(
        img_cv,
        amount,
        sigma,
        threshold,
        luminance_only=True
    ):
        if luminance_only:
            lab = cv2.cvtColor(
                img_cv,
                cv2.COLOR_BGR2LAB
            )

            l, a, b = cv2.split(lab)

            blur = cv2.GaussianBlur(
                l,
                ksize=(0, 0),
                sigmaX=sigma
            )

            diff = cv2.subtract(
                l,
                blur
            )

            if threshold > 0:
                gate = cv2.threshold(
                    cv2.absdiff(l, blur),
                    threshold,
                    255,
                    cv2.THRESH_BINARY
                )[1]

                diff = cv2.bitwise_and(
                    diff,
                    gate
                )

            l2 = cv2.add(
                l,
                cv2.convertScaleAbs(
                    diff,
                    alpha=amount
                )
            )

            output_lab = cv2.merge(
                [l2, a, b]
            )

            return cv2.cvtColor(
                output_lab,
                cv2.COLOR_LAB2BGR
            )

        blur = cv2.GaussianBlur(
            img_cv,
            ksize=(0, 0),
            sigmaX=sigma
        )

        return cv2.addWeighted(
            img_cv,
            amount,
            blur,
            -(amount - 1.0),
            0
        )

    # =========================================================
    # 見開きページを左右に分割
    # =========================================================
    def _split_two_up_pil(img_pil):
        w, h = img_pil.size
        mid = w // 2

        left = img_pil.crop(
            (0, 0, mid, h)
        )

        right = img_pil.crop(
            (mid, 0, w, h)
        )

        return [left, right]

    # =========================================================
    # PDFを400dpiのカラーPNGとして読み込む
    # =========================================================
    pil_pages = convert_from_path(
        _fpath,
        dpi=_dpi,
        fmt="png"
    )

    list_image = []

    for page in pil_pages:

        # 見開きの場合は左右分割
        if _split_two_up:
            faces = _split_two_up_pil(page)
        else:
            faces = [page]

        for face in faces:

            # 上下余白をトリミング
            face = _trim_margins(
                face,
                _trim_top_pct,
                _trim_bottom_pct
            )

            # PIL → OpenCV
            cvimg = _pil_to_cv(face)

            # 現在はFalseのため実行されない
            if _do_deskew:
                cvimg = _deskew(
                    cvimg,
                    max_angle=_deskew_max_angle
                )

            # 現在はFalseのため実行されない
            if _do_clahe:
                cvimg = _clahe(cvimg)

            # 現在はFalseのため実行されない
            if _do_unsharp:
                cvimg = _unsharp(
                    cvimg,
                    amount=_amount,
                    sigma=_sigma,
                    threshold=_threshold,
                    luminance_only=True
                )

            # カラー画像のままPNGへエンコード
            ok, buf = cv2.imencode(
                ".png",
                cvimg
            )

            if not ok:
                raise RuntimeError(
                    "OpenCVによるPNGエンコードに失敗しました。"
                )

            img_str = base64.b64encode(
                buf.tobytes()
            ).decode("utf-8")

            list_image.append(img_str)

    return list_image


# [FIX2] None 渡しを解消
image_list = pdf_to_image(_fpath)
print(f"生成した画像(Base64)の枚数: {len(image_list)}")


def Image_to_English_Text(image_list, filename_extension):

    API_select = "AzureOpenAI"

    Temp_value      = 0.0
    # [FIX6] top_p を削除
    MaxTokens_value = 8000

    OPENAI_API_KEY                    = "sk-"
    AZURE_OPENAI_API_KEY              = userdata.get('AZURE_OPENAI_API_KEY')
    AZURE_OPENAI_ENDPOINT             = userdata.get('AZURE_OPENAI_ENDPOINT')
    AZURE_OPENAI_API_VERSION          = "2025-01-01-preview"
    # [FIX5] gpt-4.1 に統一 ※ Azure のデプロイ名に合わせて変更してください
    AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = "gpt-4.1"

    if API_select == "OpenAI":
        client = OpenAI(api_key=OPENAI_API_KEY)
    else:
        client = AzureOpenAI(
                    azure_endpoint = AZURE_OPENAI_ENDPOINT,
                    api_key        = AZURE_OPENAI_API_KEY,
                    api_version    = AZURE_OPENAI_API_VERSION,
                )

    prompt_002c = """
        # Instructions
            1.Return exactly the text visible on this page.
              Do not add any explanations or metadata.

            I will explain it again.

            1.Return exactly the text visible on this page.
              Do not add any explanations or metadata.
        """

    def call_vision(image_b64: str, prompt: str) -> str:
        for i in range(5):
            try:
                resp = client.chat.completions.create(
                    model    = AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
                    messages = [
                        {
                            "role": "system",
                            "content": """
                                You are an OCR engine. Do NOT normalize, correct, or deduplicate text.
                                If the same word appears twice consecutively, output it twice.
                                Preserve punctuation, hyphens, spaces, and line breaks exactly.
                                Never omit short phrases around commas or at line ends.
                            """
                        },
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url":    f"data:image/png;base64,{image_b64}",
                                        "detail": "high"
                                    }
                                }
                            ]
                        }
                    ],
                    temperature = Temp_value,
                    # [FIX6] top_p 削除
                    max_tokens  = MaxTokens_value,
                )
                return resp.choices[0].message.content
            except Exception:
                time.sleep(1.5 * (i + 1))
        raise RuntimeError("Vision API retry exceeded")

    prompt = prompt_002c.format(_title=filename_extension)

    CONCURRENCY = 6
    sem         = Semaphore(CONCURRENCY)

    def guarded_call(img):
        with sem:
            return call_vision(img, prompt)

    results = [""] * len(image_list)

    with ThreadPoolExecutor(max_workers=6) as ex:
        futs = {ex.submit(guarded_call, img): idx for idx, img in enumerate(image_list)}
        for f in as_completed(futs):
            results[futs[f]] = f.result()

    return "\n\n".join(results)


def file_name_extraction(_fpath):
    filename = os.path.basename(_fpath)
    return os.path.splitext(filename)[0]


def file_save(context, file_name, _save_folder):
    os.makedirs(_save_folder, exist_ok=True)
    filename = os.path.join(_save_folder, file_name)
    with open(filename, "w", encoding="utf-8") as file:
        file.write(context)


filename_extension = file_name_extraction(_fpath)

English_Text_All = Image_to_English_Text(image_list, filename_extension)

elapsed = time.perf_counter() - t0
h, rem  = divmod(elapsed, 3600)
m, s    = divmod(rem, 60)
print(f"経過時間(プログラム開始→Image_to_English_Text完了): {int(h):02d}:{int(m):02d}:{s:06.3f}")

file_name = input("context_allを保存するファイル名を入力してください（例: output.txt）: ")
file_save(English_Text_All, file_name, _save_folder)
print(f"context_allが '{file_name}' に保存されました。")


# ============================================================
# Cell 3: 請求項生成
# ============================================================

def Claim_Extraction(text):

    API_select = "AzureOpenAI"

    Temp_value      = 0.0
    # [FIX6] top_p を削除
    MaxTokens_value = 8000

    OPENAI_API_KEY                    = "sk-"
    AZURE_OPENAI_API_KEY              = userdata.get('AZURE_OPENAI_API_KEY')
    AZURE_OPENAI_ENDPOINT             = userdata.get('AZURE_OPENAI_ENDPOINT')
    AZURE_OPENAI_API_VERSION          = "2025-01-01-preview"
    # [FIX5] gpt-4.1 に統一 ※ Azure のデプロイ名に合わせて変更してください
    AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = "gpt-4.1"

    if API_select == "OpenAI":
        client = OpenAI(api_key=OPENAI_API_KEY)
    else:
        client = AzureOpenAI(
                    azure_endpoint = AZURE_OPENAI_ENDPOINT,
                    api_key        = AZURE_OPENAI_API_KEY,
                    api_version    = AZURE_OPENAI_API_VERSION,
                )

    role_extraction = """
        You are a professional patent attorney and information-extraction specialist.
        Extract the Claims section from an English-language patent and return **Markdown only**.
        **Do not output JSON** or any text outside the specified sections.

        Be robust to headings like "Claims", "CLAIMS", or "What is claimed is:".

        Output exactly these sections, in this order:

        # Claims (verbatim)
        - Numbered list of all claims, **verbatim** (no translation or paraphrase).
        - Preserve original punctuation.
        - Remove line numbers, headers/footers, page breaks, and hyphenation artifacts at line breaks.
    """.strip()

    text_body   = text if text is not None else ""

    prompt_user = f"""
        Target: Extract claims and return **Markdown only** in the format specified by the system prompt.

        Text starts below:
        ---
        {text_body}
        ---
    """.strip()

    # [FIX7] リトライ追加
    for i in range(5):
        try:
            resp = client.chat.completions.create(
                model    = AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
                messages = [
                    {"role": "system", "content": role_extraction},
                    {"role": "user",   "content": prompt_user},
                ],
                temperature = Temp_value,
                # [FIX6] top_p 削除
                max_tokens  = MaxTokens_value,
            )
            return resp.choices[0].message.content
        except Exception:
            time.sleep(1.5 * (i + 1))
    raise RuntimeError("Claim_Extraction API retry exceeded")


Claims = Claim_Extraction(English_Text_All)
print(Claims)


# ============================================================
# Cell 4: 請求項 日本語訳
# ============================================================

def Claim_Translation(text):

    API_select = "AzureOpenAI"

    Temp_value      = 0.0
    # [FIX6] top_p を削除
    MaxTokens_value = 16000

    OPENAI_API_KEY                    = "sk-"
    AZURE_OPENAI_API_KEY              = userdata.get('AZURE_OPENAI_API_KEY')
    AZURE_OPENAI_ENDPOINT             = userdata.get('AZURE_OPENAI_ENDPOINT')
    AZURE_OPENAI_API_VERSION          = "2025-01-01-preview"
    # [FIX5] gpt-4.1 に統一（旧: gpt-4o-2024-11-20 から変更）
    #        ※ Azure のデプロイ名に合わせて変更してください
    AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = "gpt-4.1"

    if API_select == "OpenAI":
        client = OpenAI(api_key=OPENAI_API_KEY)
    else:
        client = AzureOpenAI(
                    azure_endpoint = AZURE_OPENAI_ENDPOINT,
                    api_key        = AZURE_OPENAI_API_KEY,
                    api_version    = AZURE_OPENAI_API_VERSION,
                )

    role_translation = """
        System Prompt — JP/US Patent Translation (EN→JA)

        You are a bilingual (Japanese/English) patent expert and professional translator
        with deep familiarity with US and JP patent practice.
        Your task is to translate English patent text into clear, natural Japanese suitable for JP filings
        while preserving the exact legal meaning.

        Goals
        1) Produce legally faithful, readable Japanese that matches JP patent drafting style.
        2) Preserve scope-defining nuances
            (open vs. closed, optional vs. mandatory, capability vs. configuration).
        3) Keep terminology consistent across the document.

        Operating Modes
        - Mode: "Claims" | "Specification" | "Abstract". Default: infer from input; if the text is a claim, use "Claims".
        - Register: formal patent style (not conversational).

        Key Conventions (must follow)
        - "comprising"               → 「〜を含む」 (open-ended)
        - "consisting of"            → 「〜からなる」 (closed)
        - "configured to"            → 「〜するように構成された」
        - "capable of"               → 「〜可能な」 (do NOT substitute for "configured to")
        - "wherein"                  → 「…において／…であって」 (choose naturally per sentence)
        - "according to claim X"     → 「請求項Xに記載の」
        - "plurality of"             → 「複数の」; "at least one" → 「少なくとも1つの」
        - "may" (permissive)         → 「〜してもよい」; "shall"/imperative in claims → 「〜すること」
        - "first/second [element]"   → 「第1の/第2の[要素]」 (labels, not ordinals)
        - Means-plus-function ("means for V-ing") → 「Vする手段」
        - "computer-readable medium" → 「コンピュータ可読媒体」
        - Avoid broadening/narrowing the scope. Do not add or omit technical content.

        Formatting & Fidelity
        - Preserve claim numbering, reference numerals, figure/step/element identifiers.
        - Keep defined terms consistent (e.g., terms in quotes or capitalized identifiers).
        - Normalize spacing, units, and punctuation for Japanese technical writing.
        - If a term is ambiguous or domain-specific, add a short bracketed translator note: 【訳注: …】 (only when necessary).

        Output
        - Return Japanese translation only by default.
        - If the input contains claims, format each claim on a new line with its original number
          (e.g., 「【請求項1】…」 or 「1. …」 depending on the source).
        - Do NOT include explanations unless asked.
        - If the source is image-only or unreadable, state: 「【訳注: 原文が判読不能のため翻訳不可】」.

        Quality Checks (silent)
        - Ensure consistent translation of repeated terms.
        - Maintain logical connectors and dependencies between claims.
        - Keep the original sentence boundaries unless unnatural in Japanese; split or merge minimally
        to improve readability without changing meaning.
    """.strip()

    text_body = text if text is not None else ""

    prompt_user = f"""
        Please translate the following English patent claims into Japanese according to the system prompt (Claims mode).
        Return Japanese only; keep numbering and legal nuance.

        Text:

        ---
        {text_body}
        ---
    """.strip()

    # [FIX7] リトライ追加
    for i in range(5):
        try:
            resp = client.chat.completions.create(
                model    = AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
                messages = [
                    {"role": "system", "content": role_translation},
                    {"role": "user",   "content": prompt_user},
                ],
                temperature = Temp_value,
                # [FIX6] top_p 削除
                max_tokens  = MaxTokens_value,
            )
            return resp.choices[0].message.content
        except Exception:
            time.sleep(1.5 * (i + 1))
    raise RuntimeError("Claim_Translation API retry exceeded")


Claims_JP = Claim_Translation(Claims)
print(Claims_JP)


Mounted at /content/drive
生成した画像(Base64)の枚数: 4
経過時間(プログラム開始→Image_to_English_Text完了): 00:00:22.598
context_allを保存するファイル名を入力してください（例: output.txt）: 260731_001.txt
context_allが '260731_001.txt' に保存されました。
# Claims (verbatim)
1. A liquid supply device, comprising:
a liquid discharge head configured to discharge liquid;
a main tank configured to store liquid to be supplied to the liquid discharge head:
a first sub-tank tank connected to the main tank and the liquid discharge head, the first sub-tank configured to store gas and liquid;
a second sub-tank connected to the first sub-tank and the liquid discharge head, the second sub-tank configured to store gas and liquid;
a first air tank connected to the first sub-tank, the first air tank configured to store gas; and
a second air tank connected to the second sub-tank, the second air tank configured to store gas, wherein
the first sub-tank is a positive-pressure sub-tank configured to supply liquid to the liquid discharge head,
the second sub-t

# 請求項生成

In [ ]:
def Claim_Extraction(text):


    #API_select ="OpenAI"
    API_select ="AzureOpenAI"

    Temp_value = 0.0
    top_p_value = 1.0
    MaxTokens_value = 8000

    # DX
    OPENAI_API_KEY = "sk-"
    AZURE_OPENAI_API_KEY = userdata.get('AZURE_OPENAI_API_KEY')
    AZURE_OPENAI_ENDPOINT = userdata.get('AZURE_OPENAI_ENDPOINT')
    AZURE_OPENAI_API_VERSION = "2025-01-01-preview"
    #AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = "gpt-4o-2024-11-20"
    AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = "gpt-4.1"

    if API_select == "OpenAI":
        client = OpenAI(
            api_key = OPENAI_API_KEY
            )
    else:
        client = AzureOpenAI(
                            azure_endpoint = AZURE_OPENAI_ENDPOINT,
                            api_key = AZURE_OPENAI_API_KEY,
                            api_version = AZURE_OPENAI_API_VERSION,
                            )

    # --- System Prompt ---
    role_extraction = """
                        You are a professional patent attorney and information-extraction specialist.
                        Extract the Claims section from an English-language patent and return **Markdown only**.
                        **Do not output JSON** or any text outside the specified sections.

                        Be robust to headings like “Claims”, “CLAIMS”, or “What is claimed is:”.

                        Output exactly these sections, in this order:

                        # Claims (verbatim)
                        - Numbered list of all claims, **verbatim** (no translation or paraphrase).
                        - Preserve original punctuation.
                        - Remove line numbers, headers/footers, page breaks, and hyphenation artifacts at line breaks.

                    """.strip()


    role_extraction_old = """
                        You are a professional patent attorney and information-extraction specialist.
                        Extract the Claims section from an English-language patent and return **Markdown only**.
                        **Do not output JSON** or any text outside the specified sections.

                        Be robust to headings like “Claims”, “CLAIMS”, or “What is claimed is:”.

                        Output exactly these sections, in this order:

                        # Claims (verbatim)
                        - Numbered list of all claims, **verbatim** (no translation or paraphrase).
                        - Preserve original punctuation.
                        - Remove line numbers, headers/footers, page breaks, and hyphenation artifacts at line breaks.

                        ## Dependency map
                        - For each claim: mark `independent` or `dependent`.
                        - List all referenced base claims; expand ranges (e.g., “claims 1–3” → 1, 2, 3).
                        - If amended/substitute/cancelled claims are present,
                          reflect the **latest operative set** and indicate status per claim when relevant.

                        ## Summary
                        - Counts: total, independent, dependent.
                        - Categories: method/process, apparatus/device/system,
                          computer-readable medium, composition/material, use, other.

                        ## Warnings
                        - Note numbering gaps/duplicates, amended/cancelled status summaries, low-confidence classifications,
                          or **claims_not_found** if no claims can be extracted (e.g., image-only pages/OCR issues).

                        Rules:
                        - Do not translate or paraphrase claim text.
                        - Keep internal reasoning hidden; output only the sections above.
                        - If uncertain about category/type, make best effort and mention the uncertainty in **Warnings**.
                    """.strip()



    # --- User Prompt（引数 text を埋め込む） ---
    text_body = text if text is not None else ""

    prompt_user = f"""
                        Target: Extract claims and return **Markdown only** in the format specified by the system prompt.

                        Text starts below:
                        ---
                        {text_body}
                        ---
                    """.strip()

    resp = client.chat.completions.create(
        model = AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": role_extraction},
            {"role": "user", "content": prompt_user},
        ],
        temperature = Temp_value,
        max_tokens = MaxTokens_value,
    )
    return resp.choices[0].message.content

Claims = Claim_Extraction(English_Text_All)

print(Claims)





# Claims (verbatim)

1. A liquid supply device, comprising:
a liquid discharge head configured to discharge liquid;
a main tank configured to store liquid to be supplied to the liquid discharge head:
a first sub-tank tank connected to the main tank and the liquid discharge head, the first sub-tank configured to store gas and liquid;
a second sub-tank connected to the first sub-tank and the liquid discharge head, the second sub-tank configured to store gas and liquid;
a first air tank connected to the first sub-tank, the first air tank configured to store gas; and
a second air tank connected to the second sub-tank, the second air tank configured to store gas, wherein
the first sub-tank is a positive-pressure sub-tank configured to supply liquid to the liquid discharge head,
the second sub-tank is a negative-pressure sub-tank configured to collect liquid from the liquid discharge head;
the first air tank is a positive-pressure air tank configured to supply a positive air pressure,
the se

# 請求項　日本語訳

In [ ]:
def Claim_Translation(text):


    #API_select ="OpenAI"
    API_select ="AzureOpenAI"

    Temp_value = 0.0
    top_p_value = 1.0
    MaxTokens_value = 16000

    # DX
    OPENAI_API_KEY = "sk-"
    AZURE_OPENAI_API_KEY = userdata.get('AZURE_OPENAI_API_KEY')
    AZURE_OPENAI_ENDPOINT = userdata.get('AZURE_OPENAI_ENDPOINT')
    AZURE_OPENAI_API_VERSION = "2025-01-01-preview"
    AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = "gpt-4o-2024-11-20"

    if API_select == "OpenAI":
        client = OpenAI(
            api_key = OPENAI_API_KEY
            )
    else:
        client = AzureOpenAI(
                            azure_endpoint = AZURE_OPENAI_ENDPOINT,
                            api_key = AZURE_OPENAI_API_KEY,
                            api_version = AZURE_OPENAI_API_VERSION,
                            )

    # --- System Prompt ---
    role_translation = """
                        System Prompt — JP/US Patent Translation (EN→JA)

                        You are a bilingual (Japanese/English) patent expert and professional translator
                        with deep familiarity with US and JP patent practice.
                        Your task is to translate English patent text into clear, natural Japanese suitable for JP filings
                        while preserving the exact legal meaning.

                        Goals
                        1) Produce legally faithful, readable Japanese that matches JP patent drafting style.
                        2) Preserve scope-defining nuances
                            (open vs. closed, optional vs. mandatory, capability vs. configuration).
                        3) Keep terminology consistent across the document.

                        Operating Modes
                        - Mode: "Claims" | "Specification" | "Abstract". Default: infer from input; if the text is a claim, use "Claims".
                        - Register: formal patent style (not conversational).

                        Key Conventions (must follow)
                        - “comprising” → 「〜を含む」 (open-ended)
                        - “consisting of” → 「〜からなる」 (closed)
                        - “configured to” → 「〜するように構成された」
                        - “capable of” → 「〜可能な」 (do NOT substitute for “configured to”)
                        - “wherein” → 「…において／…であって」 (choose naturally per sentence)
                        - “according to claim X” → 「請求項Xに記載の」
                        - “plurality of” → 「複数の」; “at least one” → 「少なくとも1つの」
                        - “may” (permissive) → 「〜してもよい」; “shall”/imperative in claims → 「〜すること」
                        - “first/second [element]” → 「第1の/第2の[要素]」 (labels, not ordinals)
                        - Means-plus-function (“means for V-ing”) → 「Vする手段」
                        - “computer-readable medium” → 「コンピュータ可読媒体」
                        - Avoid broadening/narrowing the scope. Do not add or omit technical content.

                        Formatting & Fidelity
                        - Preserve claim numbering, reference numerals, figure/step/element identifiers.
                        - Keep defined terms consistent (e.g., terms in quotes or capitalized identifiers).
                        - Normalize spacing, units, and punctuation for Japanese technical writing.
                        - If a term is ambiguous or domain-specific, add a short bracketed translator note: 【訳注: …】 (only when necessary).

                        Output
                        - Return Japanese translation only by default.
                        - If the input contains claims, format each claim on a new line with its original number
                          (e.g., 「【請求項1】…」 or 「1. …」 depending on the source).
                        - Do NOT include explanations unless asked.
                        - If the source is image-only or unreadable, state: 「【訳注: 原文が判読不能のため翻訳不可】」.

                        Quality Checks (silent)
                        - Ensure consistent translation of repeated terms.
                        - Maintain logical connectors and dependencies between claims.
                        - Keep the original sentence boundaries unless unnatural in Japanese; split or merge minimally
                        to improve readability without changing meaning.

                    """.strip()

    # --- User Prompt（引数 text を埋め込む） ---
    text_body = text if text is not None else ""

    prompt_user = f"""
                        Please translate the following English patent claims into Japanese according to the system prompt (Claims mode).
                        Return Japanese only; keep numbering and legal nuance.

                        Text:

                        ---
                        {text_body}
                        ---
                    """.strip()

    resp = client.chat.completions.create(
        model = AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": role_translation},
            {"role": "user", "content": prompt_user},
        ],
        temperature = Temp_value,
        max_tokens = MaxTokens_value,
    )
    return resp.choices[0].message.content

Claims_JP = Claim_Translation(Claims)

print(Claims_JP)



